# Results

## Get data

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.express as px
import wandb
from transformers import AutoTokenizer

from helpers import get_data_from_file

template = "plotly_white"

fancy_cols = {'corr2incorr':            {"name": "Correct ⟶ Incorrect", "asc": True},
              'peak_ram_memory_mb':     {"name": "Peak RAM memory (MB)", "asc": True},
              'accuracy_sentences':     {"name": "Accuracy (sentences)", "asc": False},
              'gpu_memory_mb':          {"name": 'GPU memory (MB)', "asc": True},
              'incorr2incorr':          {"name": "Incorrect ⟶ Incorrect", "asc": True},
              'word_incorrection_rate': {"name": "Word incorrection rate", "asc": True},
              'ms_per_sentence':        {"name": "Inference time (ms/sentence)", "asc": True},
              'throughput_words':       {"name": "Throughput (words/s)", "asc": False},
              'accuracy_words':         {"name": "Accuracy (words)", "asc": False},
              'recall':                 {"name": "Recall", "asc": False},
              'incorr2corr':            {"name": "Incorrect ⟶ Correct", "asc": False},
              'corr2corr':              {"name": "Correct ⟶ Correct", "asc": False},
              'f05':                    {"name": "F0.5", "asc": False},
              'precision':              {"name": "Precision", "asc": False},
              'model_size':             {"name": "Model size (MB)", "asc": True},
              }

api = wandb.Api()
runs = api.runs("martin-elias-ctu-fit/Benchmarks")

run_dfs = []
for run in runs:
    run_df = run.history(keys=None)
    run_df["name"] = run.name
    # Rename old run metrics from token to word
    run_df.rename(columns={
            "throughput_tokens":       "throughput_words",
            "token_incorrection_rate": "word_incorrection_rate",
            "token_correction_rate":   "word_correction_rate",
            "accuracy_tokens":         "accuracy_words",

    }, inplace=True)
    run_dfs.append(run_df)
df = pd.concat(run_dfs, axis=0)

# Drop anything I don't care about in the graph
df.drop(columns=["_runtime", "_step", "_timestamp", "model_name", "skipped", 'ram_memory_mb', "should_skip",
                 "word_correction_rate"], inplace=True)

# Fix values and rename
# --------------------------------------------------------------------------------------------------------------
# Jamspell
# --------------------------------------------------------------------------------------------------------------
df.loc[df.name == "jamspell", "model_size"] = 35
df.loc[df.name == "jamspell", "name"] = "JamSpell"
# --------------------------------------------------------------------------------------------------------------
# ELMO-SC-LSTM
# --------------------------------------------------------------------------------------------------------------
df.loc[df.name == "elmo-checker-pretrained-wo_space_correction", "name"] = "ELMO-SC-LSTM Pre-trained"
df.loc[df.name == "elmo-checker-finetuned-wo_space_correction", "name"] = "ELMO-SC-LSTM Fine-tuned"
df.loc[df.name == "elmo-checker-finetuned", "name"] = "ELMO-SC-LSTM Fine-tuned Space correction"
df.loc[df.name == "elmo-checker-pretrained", "name"] = "ELMO-SC-LSTM Pre-trained Space correction"
df.loc[
    df.name == "elmo-checker-pretrained-wo space correction-tokenized", "name"] = "ELMO-SC-LSTM Pre-trained Token based"
df.loc[
    df.name == "elmo-checker-finetuned-wo space correction-tokenized", "name"] = "ELMO-SC-LSTM Fine-tuned Token based"
# --------------------------------------------------------------------------------------------------------------
# BERT
# --------------------------------------------------------------------------------------------------------------
df.loc[df.name == "bert-checker-pretrained-wo space correction", "name"] = "BERT Pre-trained"
df.loc[df.name == "bert-checker-finetuned-wo space correction", "name"] = "BERT Fine-tuned"
df.loc[df.name == "bert-checker-finetuned", "name"] = "BERT Fine-tuned Space correction"
df.loc[df.name == "bert-checker-pretrained", "name"] = "BERT Pre-trained Space correction"
df.loc[df.name == "bert-checker-pretrained-wo space correction-tokenized", "name"] = "BERT Pre-trained Token based"
df.loc[df.name == "bert-checker-finetuned-wo space correction-tokenized", "name"] = "BERT Fine-tuned Token based"
# --------------------------------------------------------------------------------------------------------------
# BART
# --------------------------------------------------------------------------------------------------------------
df.loc[df.name == "pszemraj-bart-base-grammar-synthesis", "name"] = "BART-base Pszemraj Pre-trained"
df.loc[df.name == "pszemraj-bart-base-grammar-synthesis-finetuned", "name"] = "BART-base Pszemraj Fine-tuned"
df.loc[df.name == "oliverguhr-spelling-correction-english-base", "name"] = "BART-base Oliverguhr Pre-trained"
df.loc[df.name == "oliverguhr-spelling-correction-english-base-finetuned", "name"] = "BART-base Oliverguhr Fine-tuned"
# --------------------------------------------------------------------------------------------------------------
# T5
# --------------------------------------------------------------------------------------------------------------
df.loc[df.name == "prithivida-grammar_error_correcter_v1", "name"] = "T5-Base Prithivida"
df.loc[df.name == "prithivida-grammar_error_correcter_v1-finetuned", "name"] = "T5-Base Prithivida Fine-tuned"
df.loc[df.name == "T5", "name"] = "T5-Base Vennify"
df.loc[df.name == "T5-finetuned", "name"] = "T5-Base Vennify Fine-tuned"
df.loc[df.name == "grammarly-coedit-large", "name"] = "T5-Large Grammarly"
df.loc[
    df.name == "grammarly-coedit-large-finetuned-short-prefix", "name"] = "T5-Large Grammarly Fine-tuned short prefix"
# --------------------------------------------------------------------------------------------------------------

# df.loc[df.name == "", "name"] = ""

In [ ]:
df

In [ ]:
def get_graph(df, annots, mx="f05", my="ms_per_sentence", font_size=19, size=None, arrows=None):
    if mx == "f05":
        title = "F<sub>0.5</sub> vs "
    elif mx == "recall":
        title = "Recall vs "
    else:
        raise ValueError(f"Unsupported value: {mx}")

    if my == "ms_per_sentence":
        title += "Speed"
    elif my == "word_incorrection_rate":
        title += "Incorrection rate"
    else:
        raise ValueError(f"Unsupported value: {my}")

    if size:
        title += "<br>(point size = word accuracy, bigger is better)"

    if mx == "f05":
        xaxis_title = "F<sub>0.5</sub> (higher is better)"
    elif mx == "recall":
        xaxis_title = "Recall (higher is better)"
    else:
        raise ValueError(f"Unsupported value: {mx}")

    if my == "ms_per_sentence":
        yaxis_title = "Inference Time (ms, lower is better)"
        yaxis_range = "reversed"
    elif my == "word_incorrection_rate":
        yaxis_title = "Incorrection rate (lower is better)"
        yaxis_range = "reversed"
    else:
        raise ValueError(f"Unsupported value: {my}")

    fig = px.scatter(
            df,
            x=mx,
            y=my,
            color="name",
            size=size,
            title=f"{title}",
            template=template,
            width=960, height=540,
    )
    fig.update_layout(
            title_x=0.5,
            xaxis=dict(range=[0, None]),
            xaxis_title=xaxis_title,
            yaxis_title=yaxis_title,
            yaxis=dict(autorange=yaxis_range),
            showlegend=False,
            font=dict(size=font_size),
    )

    if size is None: fig.update_traces(marker={'size': 15})

    for i, row_name in enumerate(df['name']):
        arr = row_name in arrows if arrows else False

        fig.add_annotation(
                x=df.loc[df['name'] == row_name, mx].iloc[0],
                y=df.loc[df['name'] == row_name, my].iloc[0],
                text=annots[i][0],
                showarrow=arr,
                font=dict(size=font_size),
                xshift=annots[i][1][0] if not arr else 0,
                yshift=annots[i][1][1] if not arr else 0,
                ax=annots[i][1][0] if arr else 0,
                ay=annots[i][1][1] if arr else 0,
        )

        print(df.iloc[i]['name'])
    fig.show()
    return fig

## BERT

In [ ]:
bert_df = df[df["name"].str.contains("bert", case=False, na=False)].copy()
bert_df["name"] = bert_df["name"].str.replace("BERT ", "", case=False, regex=False).str.strip()
bert_df = bert_df.sort_values(by="name", ascending=False)

In [ ]:
for col in bert_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_bert_df = bert_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_bert_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "BERT version"},
                template=template,

        )
        fig.update_layout(title_x=0.5, xaxis_title='', showlegend=False)
        # fig.show()


In [ ]:
annots = [("Pre-trained<br>token based", (0, -40)),
          ("Pre-trained<br>space correction", (0, 40)),
          ("Pre-trained", (10, 30)),
          ("Fine-tuned<br>token based", (0, -40)),
          ("Fine-tuned<br>space correction", (80, -20)),
          ("Fine-tuned", (10, -30))]

arrows = ["Pre-trained", "Fine-tuned",]

fig = get_graph(df=bert_df, annots=annots, size="accuracy_words", arrows=arrows)
fig.write_image(file="../thesis/images/bert_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

## ELMO

In [ ]:
elmo_df = df[df["name"].str.contains("elmo", case=False, na=False)].copy()
elmo_df["name"] = elmo_df["name"].str.replace("ELMO-SC-LSTM ", "", case=False, regex=False).str.strip()
elmo_df = elmo_df.sort_values(by="name", ascending=False)

In [ ]:
annots = [("Pre-trained<br>token based", (0, -40)),
          ("Pre-trained<br>space correction", (60, 40)),
          ("Pre-trained", (0, 20)),
          ("Fine-tuned<br>token based", (-5, -40)),
          ("Fine-tuned<br>space correction", (-20, -50)),
          ("Fine-tuned", (-40, 20)), ]

arrows = ["Pre-trained Space correction", "Fine-tuned Space correction",]

fig = get_graph(df=elmo_df, annots=annots, size="accuracy_words", arrows=arrows)
fig.write_image(file="../thesis/images/elmo_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

In [ ]:
for col in elmo_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_elmo_df = elmo_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_elmo_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='', showlegend=False)
        fig.show()


## BART

In [ ]:
bart_df = df[df["name"].str.contains("bart", case=False, na=False)].copy()
bart_df["name"] = bart_df["name"].str.replace("BART-base ", "", case=False, regex=False).str.strip()
bart_df = bart_df.sort_values(by="name", ascending=False)

In [ ]:
offsets = [(0, 40), (0, -40), (0, 40), (0, -40)]
annots = [("Pszemraj<br>Fine-tuned", (10, 40)),
          ("Pszemraj<br>Pre-trained", (0, -40)),
          ("Oliverguhr<br>Pre-trained", (0, 40)),
          ("Oliverguhr<br>Fine-tuned", (0, -40)), ]
fig = get_graph(df=bart_df, annots=annots, size="accuracy_words")
fig.write_image(file="../thesis/images/bart_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

In [ ]:
for col in bart_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_bart_df = bart_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_bart_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='', showlegend=False)
        fig.show()

## T5

In [ ]:
t5_df = df[df["name"].str.contains("t5", case=False, na=False)].copy()
t5_df["name"] = t5_df["name"].str.replace("T5-", "", case=False, regex=False).str.strip()
t5_df = t5_df.sort_values(by="name", ascending=False)
t5_df

In [ ]:
annots = [("Grammarly<br>Fine-tuned<br>short prefix", (-70, 0)),
          ("Grammarly<br>Pre-trained", (20, -40)),
          ("Vennify<br>Fine-tuned", (0, -40)),
          ("Vennify<br>Pre-trained", (0, -40)),
          ("Prithivida<br>Fine-tuned", (0, -40)),
          ("Prithivida<br>Pre-trained", (0, -40)), ]

fig = get_graph(df=t5_df, annots=annots, size="accuracy_words")
fig.write_image(file="../thesis/images/t5_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

In [ ]:
for col in t5_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_t5_df = t5_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_t5_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='', showlegend=False)
        fig.show()

## ALL graphs

In [ ]:
# Get all the graphs
for col in df:
    if col in ["name", "inference_time", "throughput_sentences", 'typo_detection_model_inference_time',
               'typo_detection_model_ms_per_sentence', ]:
        continue
    sorted_df = df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
    fig = px.bar(
            sorted_df,
            x=col,
            y="name",
            color="name",
            title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
            labels={"name": "Model"},
    )
    fig.update_layout(title_x=0.5, xaxis_title='')
    # fig.show()

In [ ]:
best_options = df[df['name'].isin([
        "JamSpell",
        "BERT Fine-tuned Space correction",
        "BERT Fine-tuned Token based",
        "ELMO-SC-LSTM Fine-tuned Space correction",
        "ELMO-SC-LSTM Fine-tuned Token based",
        "BART-base Oliverguhr Fine-tuned",
        "T5-Base Prithivida Fine-tuned",

])]

In [ ]:
annots = [
        ("JamSpell", (-70, 10)),
        ("BERT", (0, -25)),
        ("BERT<br>Tokenized", (70, 0)),
        ("ELMO-SC-LSTM", (0, 25)),
        ("ELMO-SC-LSTM<br>Tokenized", (0, -40)),
        ("BART", (0, -30)),
        ("T5", (0, 30)),

]

fig = get_graph(df=best_options, annots=annots, size="accuracy_words")
fig.write_image(file="../thesis/images/models_comparison_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

In [ ]:
annots = [
        ("JamSpell", (-50, 30)),
        ("BERT", (0, -25)),
        ("BERT<br>Tokenized", (-20, -40)),
        ("ELMO-SC-LSTM", (0, 25)),
        ("ELMO-SC-LSTM<br>Tokenized", (60, 40)),
        ("BART", (20, 25)),
        ("T5", (-20, -25)),

]
arrows = ["JamSpell", "BERT Fine-tuned Token based", "ELMO-SC-LSTM Fine-tuned Token based"]

fig = get_graph(df=best_options, annots=annots, my="word_incorrection_rate", mx="recall", size=None, arrows=arrows)
fig.write_image(file="../thesis/images/models_comparison_recallVSincorr_rate.pdf", width=960, height=540, engine="kaleido")

## Detect typo tokenizer max len

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nreimers/MiniLM-L6-H384-uncased")

allDF = []
for name in ['train', 'dev', 'test']:
    df, _ = get_data_from_file(name)
    allDF.extend(df)

print(len(allDF))

token_lengths = []
start = time.time()
for text in allDF:
    tokens = tokenizer.encode(text)
    token_lengths.append(len(tokens))
print(f"Time taken to tokenize: {time.time() - start:.2f} seconds")

df_tokens = pd.DataFrame({'Token Length': token_lengths})

In [ ]:
max_len = 96
fig = px.histogram(
        df_tokens,
        x="Token Length",
        nbins=50,
        title="Distribution of Token Lengths in Dataset",
        labels={"Token Length": "Number of Tokens"},
        template=template,
)
fig.add_vline(
        x=max_len,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Current max_len: {max_len}",
        annotation_position="top right"
)
fig.update_layout(
        xaxis_title="Number of Tokens",
        yaxis_title="Number of Sentences",
        bargap=0.1
)
fig.show()

# Statistics
print(f"Maximum token length: {max(token_lengths)}")
print(f"Mean token length: {np.mean(token_lengths):.2f}")
print(f"Median token length: {np.median(token_lengths)}")
print(f"95th percentile: {np.percentile(token_lengths, 95)}")
print(f"99th percentile: {np.percentile(token_lengths, 99)}")
print(f"Percentage of sentences truncated: {sum(l > max_len for l in token_lengths) / len(token_lengths):.2%}")
